# Tutorial 7: Transfer Learning

## Compare Feature Extraction vs. Fine-Tuning

**Objectives:**

* Understand Transfer Learning Concepts
* Explore Feature Extraction
* Explore Fine-Tuning
* Work with Pretrained Models
* Model Evaluation

---

### Step 1: Import Libraries

We will import TensorFlow, Keras, and CIFAR-10 dataset utilities to get started.

In [ ]:
import tensorflow as tf
from tensorflow.keras.applications import VGG16, ResNet50
from tensorflow.keras.layers import Dense, Flatten
from tensorflow.keras.models import Model
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.utils import to_categorical

---

### Step 2: Load and Preprocess the Dataset

We'll load the CIFAR-10 dataset and normalize the pixel values to be between 0 and 1. We will also convert the labels to one-hot encoding.

In [ ]:
# Load CIFAR-10 dataset
(x_train, y_train), (x_test, y_test) = cifar10.load_data()

# Normalize pixel values
x_train = x_train.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0

# One-hot encoding for labels
y_train = to_categorical(y_train, 10)
y_test = to_categorical(y_test, 10)

170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 3s 0us/step


---

### Step 3: Load Pretrained Models

#### Feature Extraction Using Pretrained VGG16

In feature extraction, we freeze all the layers of the pretrained model and only train the final classification layer.

**1. Load Pretrained Model (VGG16):** We'll load VGG16 pretrained on ImageNet and freeze its layers.

In [ ]:
# Feature Extraction With VGG16
base_model_vgg = VGG16(weights='imagenet', include_top=False, input_shape=(32, 32, 3))

# Freeze all layers
for layer in base_model_vgg.layers:
    layer.trainable = False

58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


**2. Add New Classification Layers:** We'll add a new fully connected (Dense) layer for classification on the CIFAR-10 dataset.

In [ ]:
x = Flatten()(base_model_vgg.output)
x = Dense(512, activation='relu')(x)
output = Dense(10, activation='softmax')(x)

model_feature_extraction = Model(inputs=base_model_vgg.input, outputs=output)

model_feature_extraction.compile(optimizer='adam',
                                 loss='categorical_crossentropy',
                                 metrics=['accuracy'])

**3. Train the Model:** Train only the new layers added for classification, keeping the pretrained layers frozen.

In [ ]:
history_feature_extraction = model_feature_extraction.fit(x_train, y_train,
                                                          epochs=10,
                                                          validation_data=(x_test, y_test),
                                                          batch_size=32)

Epoch 1/10
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 673s 430ms/step - accuracy: 0.4824 - loss: 1.4713 - val_accuracy: 0.5672 - val_loss: 1.2195
Epoch 2/10
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 642s 404ms/step - accuracy: 0.5949 - loss: 1.1605 - val_accuracy: 0.5724 - val_loss: 1.2144
Epoch 3/10
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 719s 428ms/step - accuracy: 0.6196 - loss: 1.0826 - val_accuracy: 0.6016 - val_loss: 1.1317
Epoch 4/10
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 677s 425ms/step - accuracy: 0.6411 - loss: 1.0177 - val_accuracy: 0.6069 - val_loss: 1.1253
Epoch 5/10
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 638s 396ms/step - accuracy: 0.6597 - loss: 0.9664 - val_accuracy: 0.6133 - val_loss: 1.1068
Epoch 6/10
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 625s 398ms/step - accuracy: 0.6757 - loss: 0.9149 - val_accuracy: 0.6080 - val_loss: 1.1379
Epoch 7/10
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 721s 423ms/step - accuracy: 0.6918 - loss: 0.8741 - val_accuracy: 0.6272 - val_loss: 1.1124
Epoch 8/10
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 662s 424ms/step - ac

---

#### Fine-Tuning Using Pretrained ResNet50

In fine-tuning, we unfreeze some of the layers of the pretrained model and retrain them along with the new layers for the new task.

**1. Load Pretrained Model (ResNet50):** We'll load ResNet50 pretrained on ImageNet and initially freeze all its layers.

In [ ]:
# Fine-Tuning with ResNet50 (without top fully connected layers)
base_model_resnet = ResNet50(weights='imagenet', include_top=False, input_shape=(32, 32, 3))

for layer in base_model_resnet.layers:
    layer.trainable = False

94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


**2. Add New Classification Layers:** Add the new classification layers for the CIFAR-10 task.

In [ ]:
x = Flatten()(base_model_resnet.output)
x = Dense(512, activation='relu')(x)
output = Dense(10, activation='softmax')(x)

model_finetune = Model(inputs=base_model_resnet.input, outputs=output)

# Initial compile
model_finetune.compile(optimizer='adam',
                       loss='categorical_crossentropy',
                       metrics=['accuracy'])

**3. Fine-Tune the Last Few Layers:** Unfreeze the last 5 layers of the ResNet50 model for fine-tuning.

In [ ]:
# Unfreeze the last 5 layers of ResNet50
for layer in base_model_resnet.layers[-5:]:
    layer.trainable = True

# Recompile with a lower learning rate for fine-tuning
model_finetune.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
                       loss='categorical_crossentropy',
                       metrics=['accuracy'])

**4. Train the Model:** Train the model, allowing the last 5 layers and the new classification layers to update during training.

In [ ]:
history_finetune = model_finetune.fit(x_train, y_train,
                                      epochs=10,
                                      validation_data=(x_test, y_test),
                                      batch_size=32)

Epoch 1/10
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 290s 180ms/step - accuracy: 0.2490 - loss: 2.1207 - val_accuracy: 0.3429 - val_loss: 1.8384
Epoch 2/10
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 276s 177ms/step - accuracy: 0.3747 - loss: 1.7851 - val_accuracy: 0.3769 - val_loss: 1.7396
Epoch 3/10
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 325s 179ms/step - accuracy: 0.4018 - loss: 1.7060 - val_accuracy: 0.3941 - val_loss: 1.6968
Epoch 4/10
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 320s 178ms/step - accuracy: 0.4285 - loss: 1.6423 - val_accuracy: 0.4024 - val_loss: 1.6984
Epoch 5/10
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 277s 177ms/step - accuracy: 0.4380 - loss: 1.6063 - val_accuracy: 0.3968 - val_loss: 1.6787
Epoch 6/10
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 321s 176ms/step - accuracy: 0.4514 - loss: 1.5731 - val_accuracy: 0.4165 - val_loss: 1.6430
Epoch 7/10
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 282s 181ms/step - accuracy: 0.4633 - loss: 1.5474 - val_accuracy: 0.4280 - val_loss: 1.6116
Epoch 8/10
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 278s 178ms/step - ac

---

### Step 4: Evaluate Both Models

We'll now evaluate both the feature extraction model and the fine-tuned model on the test set and compare their performance.

In [ ]:
# Evaluate feature extraction model
loss_fe, acc_feature_extraction = model_feature_extraction.evaluate(x_test, y_test)

# Evaluate fine-tuned model
loss_ft, acc_finetune = model_finetune.evaluate(x_test, y_test)

313/313 ━━━━━━━━━━━━━━━━━━━━ 104s 331ms/step - accuracy: 0.6317 - loss: 1.1069
313/313 ━━━━━━━━━━━━━━━━━━━━ 40s 128ms/step - accuracy: 0.4466 - loss: 1.5552


---

### Step 5: Compare Results

Finally, compare the results of both models:

In [ ]:
print(f"Feature Extraction Accuracy: {acc_feature_extraction * 100:.2f}%")
print(f"Fine-Tuning Accuracy: {acc_finetune * 100:.2f}%")

---

# **Tasks:**

1. **PyTorch Implementation:** Try implementing this entire tutorial using PyTorch.
2. **Improve Fine-Tuning:** How would you improve results?
* Changing learning rate
* Preventing overfitting (Dropout, Augmentation)
* Early stopping
* Preprocessing/formatting adjustments
* Adjusting the number of epochs
* Unfreezing more layers or different layers


3. **Cross-Comparison:** * Do feature extraction with ResNet50 and compare with VGG16 and a custom model.
* Perform fine-tuning with VGG16 and feature extraction with ResNet50.




# **Solution for Task 1,2,3**
This  code block contains the full **PyTorch implementation** for all tasks. It includes **Data Augmentation for preprocessing, an Early Stopping class to prevent overfitting, and a Learning Rate Scheduler.** It compares a **Custom CNN with ResNet50 Feature Extraction and VGG16 Fine-Tuning,** using different learning rates and unfreezing specific layers for the best results.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torchvision import models
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

# 1. Configuration & Hyperparameters [cite: 70, 71, 75]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
batch_size = 64
epochs = 30
base_lr = 0.001
finetune_lr = 1e-5 # Reduced learning rate for fine-tuning [cite: 57, 71]

# 2. Preprocessing & Data Augmentation (Prevent Overfitting) [cite: 72, 74]
transform_train = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
])

transform_test = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
])

trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform_train)
trainloader = DataLoader(trainset, batch_size=batch_size, shuffle=True)
testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform_test)
testloader = DataLoader(testset, batch_size=batch_size, shuffle=False)

# 3. Early Stopping Class [cite: 73]
class EarlyStopping:
    def __init__(self, patience=5, min_delta=0.001):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_acc = None
        self.early_stop = False

    def __call__(self, val_acc):
        if self.best_acc is None:
            self.best_acc = val_acc
        elif val_acc < self.best_acc + self.min_delta:
            self.counter += 1
            if self.counter >= self.patience: self.early_stop = True
        else:
            self.best_acc = val_acc
            self.counter = 0

# 4. Model Definitions [cite: 80, 81, 82]

# Custom CNN for comparison [cite: 81]
class CustomCNN(nn.Module):
    def __init__(self):
        super(CustomCNN, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2, 2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2, 2)
        )
        self.classifier = nn.Sequential(nn.Flatten(), nn.Linear(64*8*8, 512), nn.ReLU(), nn.Linear(512, 10))
    def forward(self, x): return self.classifier(self.features(x))

# ResNet50 Feature Extraction [cite: 80, 82]
def get_resnet_fe():
    model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
    for param in model.parameters(): param.requires_grad = False # Freeze layers [cite: 28, 47]
    model.fc = nn.Sequential(nn.Dropout(0.3), nn.Linear(model.fc.in_features, 512), nn.ReLU(), nn.Linear(512, 10))
    return model.to(device)

# VGG16 Fine-Tuning [cite: 82]
def get_vgg_ft():
    model = models.vgg16(weights=models.VGG16_Weights.IMAGENET1K_V1)
    for param in model.parameters(): param.requires_grad = False
    # Unfreeze more layers (last 2 blocks) [cite: 55, 56, 77, 79]
    for param in model.features[-7:].parameters(): param.requires_grad = True
    model.classifier[6] = nn.Linear(model.classifier[6].in_features, 10)
    return model.to(device)

# 5. Generic Training & Plotting Loop [cite: 41, 59, 60]
def train_and_plot(name, model, lr):
    print(f"\n--- Training {name} ---")
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=lr)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'max', patience=2)
    early_stopper = EarlyStopping(patience=5)

    history = {'loss': [], 'acc': []}

    for epoch in range(epochs):
        model.train()
        epoch_loss = 0.0
        for inputs, labels in trainloader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()

        # Evaluation [cite: 61, 62, 63]
        model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for inputs, labels in testloader:
                inputs, labels = inputs.to(device), labels.to(device)
                _, pred = torch.max(model(inputs), 1)
                total += labels.size(0)
                correct += (pred == labels).sum().item()

        accuracy = 100 * correct / total
        history['loss'].append(epoch_loss/len(trainloader))
        history['acc'].append(accuracy)
        print(f"Epoch {epoch+1}: Loss {history['loss'][-1]:.4f}, Acc {accuracy:.2f}%")

        scheduler.step(accuracy)
        early_stopper(accuracy)
        if early_stopper.early_stop: break

    return history

# 6. Run Experiments and Visualize Results [cite: 64, 65, 66]
results = {
    "Custom CNN": train_and_plot("Custom CNN", CustomCNN().to(device), base_lr),
    "ResNet50 FE": train_and_plot("ResNet50 FE", get_resnet_fe(), base_lr),
    "VGG16 FT": train_and_plot("VGG16 FT", get_vgg_ft(), finetune_lr)
}

plt.figure(figsize=(12, 5))
for name, data in results.items():
    plt.plot(data['acc'], label=name)
plt.title('Model Accuracy Comparison')
plt.xlabel('Epochs')
plt.ylabel('Accuracy (%)')
plt.legend()
plt.show()

100%|██████████| 170M/170M [00:02<00:00, 69.0MB/s]



--- Training Custom CNN ---
Epoch 1: Loss 1.2887, Acc 63.57%
Epoch 2: Loss 0.9565, Acc 68.49%
Epoch 3: Loss 0.8284, Acc 71.26%
Epoch 4: Loss 0.7364, Acc 73.58%
Epoch 5: Loss 0.6682, Acc 74.24%
Epoch 6: Loss 0.6067, Acc 74.42%
Epoch 7: Loss 0.5551, Acc 74.84%
Epoch 8: Loss 0.5111, Acc 76.53%
Epoch 9: Loss 0.4773, Acc 76.03%
Epoch 10: Loss 0.4420, Acc 76.92%
Epoch 11: Loss 0.4016, Acc 76.73%
Epoch 12: Loss 0.3773, Acc 76.87%
Epoch 13: Loss 0.3564, Acc 76.53%
Epoch 14: Loss 0.2478, Acc 78.56%
Epoch 15: Loss 0.2270, Acc 78.87%
Epoch 16: Loss 0.2136, Acc 79.04%
Epoch 17: Loss 0.1986, Acc 78.71%
Epoch 18: Loss 0.1924, Acc 79.31%
Epoch 19: Loss 0.1851, Acc 78.98%
Epoch 20: Loss 0.1779, Acc 79.21%
Epoch 21: Loss 0.1674, Acc 79.10%
Epoch 22: Loss 0.1573, Acc 79.21%
Epoch 23: Loss 0.1570, Acc 79.26%
Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 127MB/s]



--- Training ResNet50 FE ---
Epoch 1: Loss 1.7688, Acc 42.64%
Epoch 2: Loss 1.6576, Acc 46.59%
Epoch 3: Loss 1.6332, Acc 45.78%
Epoch 4: Loss 1.6122, Acc 46.59%
Epoch 5: Loss 1.6018, Acc 47.27%


as we can see that the trend is showing increasing behavior..This process is taking too much time so i am stopping right here..